# Chit 6 baseline: a heartbeat classifier for a wearable · **MCU**
**AIMS Edge ML for Science · Capstone baseline** · 🟩 AI-on

Train a small 1-D CNN on single beats, measure macro-F1 on **unseen patients** in fp32 and with int8 weights and activations, and count what it costs on a microcontroller: flash, SRAM, MACs, and energy per beat.

This is a **starting point, not a good design**: a small, simple model that runs end to end (data → training → the chit's metric → its cost on the target),
so your team has something to beat. Every number it prints is yours to improve. Run time: ~3 min on a laptop CPU (measured), plus the first download; training uses the GPU automatically if the runtime has one.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order.

In [ ]:
# --- Colab setup: run once per session (~1 min) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the chit's data on your Google Drive between sessions
LAB = "capstone/baselines"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["onnx", "onnxruntime", "wfdb"]], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import os, sys, time, json, warnings
sys.path.append("../.."); warnings.filterwarnings("ignore")
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common.budget import profile, fits
from capstone.baselines.tools import FAST, fit, predict, to_onnx, ort_session, ort_int8, ort_predict, ort_latency, mcu_latency, energy_uj, report
torch.manual_seed(0); np.random.seed(0)
os.makedirs("my_models", exist_ok=True)
T0 = time.time()

## 0. Understand the data
Before any model, look at the data: what one sample is, what the classes look like, how the splits were made, and why the chit uses its metric.
This part only reads and plots (≲ 1 min). The model starts in Part 1.

### 0.1 The task in plain words
An electrocardiogram (ECG) records the electrical activity of the heart through electrodes on the skin. A small patch worn for several days could watch every
heartbeat and flag the abnormal ones, so that a doctor sees only the interesting moments. It runs on a coin cell, so the classifier must be tiny.
**One decision of the model** = the class of one heartbeat: normal, or one of four kinds of abnormal beat.

### 0.2 What one sample is
One sample is **one heartbeat**: 256 numbers = **0.71 s** of one ECG lead (MLII) at 360 samples per second, in **millivolts**, unfiltered.
The window is centred on the beat's **R peak** (the tall spike of the beat, at sample 128), as marked by cardiologists.
**Label** (AAMI groups): **N** = normal (including bundle-branch-block beats: the beat starts normally but travels slowly);
**S** = supraventricular ectopic: an early beat that starts in the upper chambers (atria), usually with a normal shape;
**V** = ventricular ectopic: a beat that starts in the lower chambers (ventricles), usually wide and oddly shaped;
**F** = fusion: a normal and a ventricular beat at the same time; **Q** = paced (made by a pacemaker) or unclassifiable.

In [ ]:
import matplotlib.pyplot as plt
from capstone.data.mitbih import load, records, CLASSES, RECORDS
FS = 360                                          # samples per second
SPLITS0 = {name: load(name) for name in ("train", "val", "test")}
X0, y0 = SPLITS0["train"]
print("one beat:", X0.shape[1:], X0.dtype, f"= 256 samples = {256 / FS:.2f} s of one ECG lead, in millivolts; the R peak is sample 128")
print("classes:", ", ".join(f"{k} = {c}" for k, c in enumerate(CLASSES)))
print(f"values (train): {X0.min():.2f} to {X0.max():.2f} mV; 99% of them between {np.quantile(X0, 0.005):.2f} and {np.quantile(X0, 0.995):.2f} mV")

In [ ]:
names = {"N": "N normal", "S": "S supraventricular", "V": "V ventricular", "F": "F fusion", "Q": "Q paced / unknown"}
amp = X0[:, 0, 128] - np.median(X0[:, 0], 1)                         # the R peak's height above the beat's baseline
width = (np.abs(X0[:, 0] - np.median(X0[:, 0], 1, keepdims=True)) > 0.5 * np.abs(amp)[:, None]).sum(1) / FS * 1000
fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
x = np.arange(3)
for k, c in enumerate(CLASSES):
    ax[0].bar(x + 0.16 * (k - 2), [np.sum(SPLITS0[s][1] == k) for s in ("train", "val", "test")], 0.16, label=names[c])
    ax[1].hist(amp[y0 == k], np.linspace(-3, 4, 71), density=True, histtype="step", lw=1.5, label=names[c])
    ax[2].hist(width[y0 == k], np.linspace(0, 400, 41), density=True, histtype="step", lw=1.5, label=names[c])
ax[0].set(xticks=x, xticklabels=["train", "val", "test"], ylabel="beats", yscale="log", title="Beats per class and split")
ax[1].set(xlabel="R-peak height above the beat's median [mV]", ylabel="density", title="Beat amplitude (train)")
ax[2].set(xlabel="time the beat is above half its peak height [ms]", ylabel="density", title="Beat width (train)")
for a in ax:
    a.legend(fontsize=7)
plt.tight_layout(); plt.show()
for k, c in enumerate(CLASSES):
    print(f"{names[c]:<19} median amplitude {np.median(amp[y0 == k]):5.2f} mV | median width {np.median(width[y0 == k]):4.0f} ms")

**What to notice.** Left (log scale): N dominates, and the rare classes are really rare: 543 S beats and 401 F beats for training, and only **14 F beats** in `val`.
Middle: the amplitude curves are spiky, because each spike is one patient: the size of a beat depends more on the patient and the electrode position than on the class.
For example, the S peak near −1.3 mV comes from one record (207) whose R peak points downwards. Right: **width** is a better clue: V beats are about three times wider
than N beats (median ~97 ms vs. ~33 ms), the textbook sign of a ventricular beat, and paced (Q) beats are wide too. S beats are as narrow as N beats:
what makes them abnormal is mainly that they come too early, and a 0.71-s window shows little of the timing. That is one reason S is hard.

### 0.3 Look at it
Five random training beats of each class on one time axis (0 = the R peak).

In [ ]:
t = (np.arange(256) - 128) / FS
rng = np.random.default_rng(0)
fig, ax = plt.subplots(1, 5, figsize=(18, 3.6), sharey=True)
for k, c in enumerate(CLASSES):
    for i in rng.choice(np.where(y0 == k)[0], 5, replace=False):
        ax[k].plot(t, X0[i, 0], lw=1)
    ax[k].axvline(0, color="k", ls=":", lw=0.8)
    ax[k].set(title=f"{names[c]} ({np.sum(y0 == k):,} in train)", xlabel="time from the R peak [s]")
ax[0].set_ylabel("ECG [mV]")
plt.tight_layout(); plt.show()

**What to notice.** N beats are a narrow spike, but the same class looks different from beat to beat: the baseline level differs (from about −1 to −0.4 mV in these five),
and the spike can be tall or small. The S beats look much like N beats. V beats are wide and their shapes differ a lot (different starting points in the ventricles).
The five F beats look alike, because almost all training F beats come from one patient (see 0.4). Paced beats (Q) start with a very short, sharp pacemaker spike.

### 0.4 The numbers that matter

In [ ]:
print("split  records    beats      " + "        ".join(CLASSES))
for name, (X, y) in SPLITS0.items():
    print(f"{name:<6} {len(RECORDS[name]):>7}  {len(y):>7,}  " + "  ".join(f"{np.mean(y == k):7.2%}" for k in range(5)))
print("\nwhere the rare classes come from (the record with the most beats of the class):")
for name, (X, y) in SPLITS0.items():
    r = records(name)
    for k in (1, 3):
        u, n = np.unique(r[y == k], return_counts=True)
        print(f"  {name:<5} {CLASSES[k]}: {n.sum():>5,} beats, {n.max() / n.sum():4.0%} of them from record {u[n.argmax()]}")

**Splits by patient.** A wearable meets patients it was never trained on, so no patient is in two splits. We start from the standard inter-patient split
(de Chazal et al., 2004: "DS1" for training, "DS2" for testing) and keep the paced records, so that class Q exists; `val` is four training-side records chosen so that it
contains every class. Records 201 and 202 are the same patient and are both in train.
**Class balance:** N is 84–85% of the beats in train and test, but only 64% in `val`, which has more V and Q. The table above shows the real problem: **each rare class comes
mostly from one patient.** 93% of the training F beats come from record 208, 94% of the test F beats from record 213, and 78% of the test S beats from record 232.
So the F1 on F and S largely measures how well a model does on one or two people, and small changes can move macro-F1 a lot: compare designs with several seeds.
**Value range:** −5.12 to +5.11 mV (the limits of the recorder; very few beats reach them), and 99% of the values are between about −2 and +1.7 mV.

### 0.5 The metric, explained on the data
**Macro-F1** is the F1 score (the balance of precision and recall) computed for each of the five classes, then averaged with **equal weight**.
Accuracy would reward ignoring the rare beats: a model that always says "N" is 85% accurate on test, but its macro-F1 is only about 0.18.
The abnormal beats (S, V, F) are the ones a doctor needs to see, so each class counts as much as N, however rare it is.

In [ ]:
del SPLITS0, X0

## 1. Data
One beat = 256 samples (0.71 s at 360 Hz) of lead MLII around the R peak. Classes N, S, V, F, Q (AAMI). No patient is in two splits.

In [ ]:
from capstone.data.mitbih import load, CLASSES, macro_f1
X_tr, y_tr = load("train"); X_va, y_va = load("val"); X_te, y_te = load("test")
for name, y in (("train", y_tr), ("val", y_va), ("test", y_te)):
    print(f"{name:<5} {len(y):>6,} beats  " + "  ".join(f"{c} {np.mean(y == k):6.2%}" for k, c in enumerate(CLASSES)))

We subtract each beat's mean (removes the slow baseline drift; 256 additions on the device, negligible).

In [ ]:
prep = lambda X: (X - X.mean(-1, keepdims=True)).astype(np.float32)
XTR, XVA, XTE = prep(X_tr), prep(X_va), prep(X_te)
if FAST:
    keep = np.random.default_rng(0).permutation(len(XTR))[:10_000]; XTR, y_tr = XTR[keep], y_tr[keep]

## 2. The model, costed before training
Three strided 1-D convolutions, global average pooling, one linear layer.

In [ ]:
model = nn.Sequential(nn.Conv1d(1, 8, 7, stride=2, padding=3), nn.ReLU(),
                      nn.Conv1d(8, 16, 5, stride=2, padding=2), nn.ReLU(),
                      nn.Conv1d(16, 32, 5, stride=2, padding=2), nn.ReLU(),
                      nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(32, 5))
p = profile(model, (1, 256))
TARGET = dict(max_weight_bytes=64 * 1024, max_act_bytes=16 * 1024, mac_rate=100e6)   # DRAFT values from chits.md; add max_latency_s once ⟨T⟩ is set
print(f"params {p['params']:,} | MACs {p['macs']:,} | int8 weights {p['weight_bytes']:,} B | peak activations {p['peak_act_bytes']:,} B")
print(f"latency at 100 M MAC/s: {1e3 * mcu_latency(p['macs'], 100e6):.3f} ms per beat | energy per beat (card formula): {energy_uj(p):.2f} µJ")
fits(p, TARGET);

## 3. Train (class-weighted: N is ~85% of the beats) and measure

In [ ]:
w = 1 / np.sqrt(np.bincount(y_tr, minlength=5) + 1); w = w / w.mean()
fit(model, XTR, y_tr, epochs=2 if FAST else 15, bs=256, lr=3e-3, class_weight=w)
F1_32 = macro_f1(y_te, predict(model, XTE).argmax(1))
print(f"macro-F1 on test patients, fp32: {F1_32:.4f}   (val: {macro_f1(y_va, predict(model, XVA).argmax(1)):.4f})")

## 4. int8 (weights and activations), with ONNX Runtime static quantization
An MCU runs int8: this is the number the chit counts.

In [ ]:
to_onnx(model.cpu(), XTR[:1], "my_models/mitbih_fp32.onnx")
ort_int8("my_models/mitbih_fp32.onnx", XTR[np.random.default_rng(0).permutation(len(XTR))[:512]], "my_models/mitbih_int8.onnx")
sess8 = ort_session("my_models/mitbih_int8.onnx")
OUT8 = ort_predict(sess8, XTE)

F1_8 = macro_f1(y_te, OUT8.argmax(1))
from sklearn.metrics import f1_score
print(f"macro-F1 on test patients, int8: {F1_8:.4f}")
print("per-class F1 (int8):", dict(zip(CLASSES, np.round(f1_score(y_te, OUT8.argmax(1), labels=range(5), average=None, zero_division=0), 3))))

In [ ]:
E = energy_uj(p)
ROW = report("6 mitbih", (len(XTR), len(XVA), len(XTE)), (1, 256), f"macro-F1 int8 {F1_8:.4f} (fp32 {F1_32:.4f})", p,
             f"MCU counted: {1e3 * mcu_latency(p['macs'], 100e6):.3f} ms at 100 M MAC/s; energy {E:.2f} µJ/beat",
             "ODC-By 1.0 (PhysioNet, confirmed)", extra=dict(energy_uj=round(E, 3), minutes=round((time.time() - T0) / 60, 1)))

---
**Where next.** Read your chit card and `targets.md` again with these numbers in front of you: which budget binds first? The levers are listed in `targets.md`.